<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎵 AceStep 1.5 XL Turbo (4B)</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle T4 GPU Edition — Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Wan2GP Engine + mmgp Offloading | 2×T4 15GB VRAM | INT8 Quantized</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-2%C3%97T4-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### Quick Start
1. **Settings → Accelerator → GPU T4 x2** (or T4 x1)
2. **Turn on Internet**
3. Run all cells in order
4. Use the Gradio public link from Cell 5 to generate music

| Feature | Details |
|---|---|
| Model | AceStep 1.5 XL Turbo 4B (quanto int8) |
| Engine | Wan2GP + mmgp Profile 2 |
| VRAM | Fits in 15GB T4 VRAM |
| Inference Steps | 8 (distilled turbo) |
| Mode | Text (Lyrics) → Music |

## ⚙️ Cell 1 — Environment Setup

In [ ]:
# Cell 1: Environment Setup
import os, gc, psutil

print("=== Kaggle T4 Environment Setup ===")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available")

os.system("echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
os.system("echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1")
gc.collect()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True)
    print("\n✅ GPU Active!")
except Exception:
    print("⚠️ No GPU detected. Go to Settings → Accelerator → GPU T4 x1")

print("✅ Environment optimized!")


## 📦 Cell 2 — Install Dependencies

In [ ]:
# Cell 2: Clone Wan2GP & Install Dependencies
!git clone https://github.com/DeepBeepMeep/Wan2GP.git
!pip install --timeout 120 --retries 5 -q -r Wan2GP/requirements.txt
!pip install --timeout 120 --retries 5 -q mmgp gradio fastapi uvicorn python-multipart pyngrok
print("\n✅ Dependencies installed!")


## 📥 Cell 3 — Download Models

In [ ]:
# Cell 3: Download All Required Models
import os, shutil
from huggingface_hub import hf_hub_download

REPO = "DeepBeepMeep/TTS"
MODEL_DIR = "Wan2GP/models"
TMP_DIR = "/kaggle/tmp/models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# === 1. XL Transformer (quanto int8 — 5.51 GB) → /kaggle/tmp ===
TRANSFORMER_FILE = "ace_step_v1_5_xl_transformer_quanto_bf16_int8.safetensors"
dest = os.path.join(MODEL_DIR, TRANSFORMER_FILE)
if not os.path.exists(dest):
    print(f"Downloading {TRANSFORMER_FILE} → /kaggle/tmp ...")
    hf_hub_download(repo_id=REPO, filename=TRANSFORMER_FILE, local_dir=TMP_DIR)
    os.symlink(os.path.join(TMP_DIR, TRANSFORMER_FILE), dest)
    print(f"  ✓ {TRANSFORMER_FILE} (symlinked)")
else:
    print(f"  ✓ Already exists: {TRANSFORMER_FILE}")

# === 2. VAE + silence_latent (ace_step15 folder — ~341 MB) ===
ACE_STEP15_FILES = [
    "ace_step15/ace_step_v1_5_audio_vae_bf16.safetensors",
    "ace_step15/silence_latent.pt",
]
for f in ACE_STEP15_FILES:
    dest = os.path.join(MODEL_DIR, f)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest):
        print(f"  ✓ Already exists: {f}")
        continue
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=REPO, filename=f, local_dir=MODEL_DIR)
    print(f"  ✓ {f}")

# === 3. Qwen3-Embedding-0.6B text encoder (~1.2 GB) ===
QWEN3_FOLDER = "Qwen3-Embedding-0.6B"
QWEN3_FILES = [
    "model.safetensors",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]
for gf in QWEN3_FILES:
    dest = os.path.join(MODEL_DIR, QWEN3_FOLDER, gf)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest):
        print(f"  ✓ Already exists: {QWEN3_FOLDER}/{gf}")
        continue
    print(f"Downloading {QWEN3_FOLDER}/{gf}...")
    hf_hub_download(repo_id=REPO, filename=f"{QWEN3_FOLDER}/{gf}", local_dir=MODEL_DIR)
    print(f"  ✓ {QWEN3_FOLDER}/{gf}")

# === Clean HF cache ===
for cache_dir in [os.path.join(MODEL_DIR, ".cache"), os.path.join(TMP_DIR, ".cache")]:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)

os.system("df -h /kaggle/working /kaggle/tmp")
print("\n✅ All model downloads complete! (~7 GB total)")


## 📝 Cell 4 — Write API Server Script (Ngrok)


In [ ]:
%%writefile run_acestep_xl.py
import gc
import os
import sys
import random
import tempfile
import traceback
import numpy as np
import subprocess
import psutil

# ---- Bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import gradio as gr

# ==== GPU INFO ====
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute Capability: {torch.cuda.get_device_capability()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

# ==== Force attention backends for T4 (SM 7.5, no flash attn) ====
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# ==== LOAD MODEL VIA WAN2GP ====
print("\nLoading AceStep 1.5 XL Turbo (quanto int8)...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl

fl.set_checkpoints_paths(["models", "ckpts", "."])

from models.TTS.ace_step_handler import family_handler

base_model_type = "ace_step_v1_5_xl"
model_def = {
    "ace_step15_transformer_variant": "xl_turbo",
    "text_encoder_folder": "acestep-5Hz-lm-1.7B",
}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

transformer_path = os.path.join("models", "ace_step_v1_5_xl_transformer_quanto_bf16_int8.safetensors")
if not os.path.isfile(transformer_path):
    raise FileNotFoundError(f"Transformer not found at {transformer_path}")
print(f"  Transformer: {os.path.basename(transformer_path)}")
sys.stdout.flush()

ace_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="ace_step_v1_5_xl",
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=torch.bfloat16,
    VAE_dtype=torch.float32,
    text_encoder_filename=None,
)

# ==== Verify pipeline components ====
print("\n--- Pipeline Components ---")
pipe_inner = pipe.get("pipe", pipe) if isinstance(pipe, dict) and "pipe" in pipe else pipe
for name, component in pipe_inner.items():
    if component is not None:
        ctype = type(component).__name__
        print(f"  {name}: {ctype}")
    else:
        print(f"  {name}: None")
sys.stdout.flush()

# ==== Apply mmgp — higher VRAM budgets for T4 15GB ====
print("\nApplying mmgp custom profile (pinned + async + HIGH budget)...")
sys.stdout.flush()

offload.profile(
    pipe_inner,
    profile_no=2,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.bfloat16,
    pinnedMemory=True,
    asyncTransfers=True,
    budgets={
        "transformer":  10000,
        "text_encoder_2": 2500,
        "codec":         2000,
        "*":             1000,
    },
)
print("✅ mmgp offloading ready (high VRAM budget)!")
sys.stdout.flush()

offload.shared_state["_attention"] = "sdpa"

# ==== Fix: re-promote XL tokenizer quantizer to float32 ====
try:
    quantizer = ace_model.ace_step_transformer.tokenizer.quantizer
    quantizer.float()
    print("✅ XL tokenizer quantizer re-promoted to float32")
except Exception as e:
    print(f"⚠️ Could not re-promote quantizer: {e}")
sys.stdout.flush()

print("\n✅ AceStep 1.5 XL Turbo loaded! Ready to generate music.")
sys.stdout.flush()

# ==== CONSTANTS ====
KEYSCALE_OPTIONS = [
    "", "C major", "C minor", "C# major", "C# minor", "Cb major", "Cb minor",
    "D major", "D minor", "D# major", "D# minor", "Db major", "Db minor",
    "E major", "E minor", "E# major", "E# minor", "Eb major", "Eb minor",
    "F major", "F minor", "F# major", "F# minor", "Fb major", "Fb minor",
    "G major", "G minor", "G# major", "G# minor", "Gb major", "Gb minor",
    "A major", "A minor", "A# major", "A# minor", "Ab major", "Ab minor",
    "B major", "B minor", "B# major", "B# minor", "Bb major", "Bb minor",
]

LANGUAGE_OPTIONS = [
    "", "en", "zh", "ja", "ko", "fr", "de", "es", "pt", "it", "ru",
    "ar", "hi", "th", "vi", "id", "tr", "pl", "nl", "sv", "fi",
    "da", "no", "cs", "ro", "hu", "el", "he", "fa", "ur", "bn",
    "ta", "te", "unknown",
]

AUDIO_TASK_OPTIONS = [
    "Text (Lyrics) → Audio",
    "Cover Mode of Source Audio",
    "Transfer Reference Audio Timbre",
    "Cover + Transfer Timbre",
]

# ==== SINGLE GENERATION (unchanged core logic) ====
@torch.inference_mode()
def _generate_single(
    lyrics, caption, duration, seed,
    num_steps, guidance_scale, temperature,
    bpm, keyscale, timesignature, language,
    negative_prompt, audio_scale, shift, infer_method,
    audio_ref, audio_ref_timbre, audio_task,
    progress_fn=None, progress_offset=0.0, progress_scale=1.0,
    track_label="",
):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    if seed is None or seed < 0:
        seed = random.randint(0, 2**32 - 1)
    seed = int(seed)

    duration = max(5, min(360, int(duration)))
    num_steps = max(1, int(num_steps))

    custom_settings = {}
    if bpm is not None and str(bpm).strip():
        try:
            bpm_val = int(float(bpm))
            if 30 <= bpm_val <= 300:
                custom_settings["bpm"] = bpm_val
        except (ValueError, TypeError):
            pass
    if keyscale and str(keyscale).strip():
        custom_settings["keyscale"] = str(keyscale).strip()
    if timesignature and str(timesignature).strip():
        try:
            ts_val = int(float(timesignature))
            if ts_val in (2, 3, 4, 6):
                custom_settings["timesignature"] = ts_val
        except (ValueError, TypeError):
            pass
    if language and str(language).strip():
        custom_settings["language"] = str(language).strip().lower()

    audio_guide = None
    audio_guide2 = None
    audio_prompt_type = ""

    task = str(audio_task or "")
    if "Cover" in task and "Timbre" in task:
        audio_prompt_type = "AB"
    elif "Cover" in task:
        audio_prompt_type = "A"
    elif "Timbre" in task:
        audio_prompt_type = "B"

    if "A" in audio_prompt_type and audio_ref is not None and os.path.isfile(str(audio_ref)):
        audio_guide = str(audio_ref)
    elif "A" in audio_prompt_type:
        return None, "❌ Cover Mode requires a Source Audio file."

    if "B" in audio_prompt_type and audio_ref_timbre is not None and os.path.isfile(str(audio_ref_timbre)):
        audio_guide2 = str(audio_ref_timbre)
    elif "B" in audio_prompt_type:
        return None, "❌ Timbre Transfer requires a Timbre Reference file."

    neg = str(negative_prompt or "").strip()
    if not neg:
        neg = "NO USER INPUT"

    free_vram = torch.cuda.mem_get_info()[0] / 1024**3
    ram = psutil.virtual_memory()
    print(f"\n{'='*60}")
    print(f"{track_label} | Task: {audio_task}")
    print(f"Generating: {duration}s, {num_steps} steps, seed={seed}")
    print(f"Lyrics: {lyrics[:80]}...")
    print(f"Caption: {caption[:80]}...")
    if custom_settings:
        print(f"Custom: {custom_settings}")
    print(f"  VRAM free: {free_vram:.2f} GB | RAM free: {ram.available / 1024**3:.1f} GB")
    print(f"{'='*60}")
    sys.stdout.flush()

    total_steps = [num_steps]
    current_step = [0]

    def cb(step_idx=-1, latent=None, force_refresh=True, read_state=False,
           override_num_inference_steps=-1, pass_no=-1, preview_meta=None,
           denoising_extra="", progress_unit=None, **kwargs):
        if override_num_inference_steps is not None and override_num_inference_steps > 0:
            total_steps[0] = override_num_inference_steps
        if step_idx is not None and step_idx >= 0:
            current_step[0] = step_idx + 1
            frac = current_step[0] / max(total_steps[0], 1)
            free_v = torch.cuda.mem_get_info()[0] / 1024**3
            print(f"  {track_label} Step {current_step[0]}/{total_steps[0]} | VRAM free: {free_v:.2f} GB")
            sys.stdout.flush()
            if progress_fn is not None:
                scaled = progress_offset + min(frac * 0.9, 0.95) * progress_scale
                progress_fn(min(scaled, 0.99), desc=f"{track_label}: Step {current_step[0]}/{total_steps[0]}")

    result = ace_model.generate(
        lyrics,
        0,
        audio_guide,
        alt_prompt=caption if caption and caption.strip() else None,
        audio_guide2=audio_guide2,
        audio_prompt_type=audio_prompt_type,
        temperature=float(temperature),
        duration_seconds=duration,
        num_inference_steps=num_steps,
        guidance_scale=float(guidance_scale),
        seed=seed,
        custom_settings=custom_settings if custom_settings else None,
        lm_negative_prompt=neg,
        audio_scale=float(audio_scale),
        shift=float(shift),
        infer_method=str(infer_method),
        callback=cb,
    )

    if result is None:
        return None, "❌ Generation failed (returned None)."

    audio_data = None
    audio_sr = 48000

    if isinstance(result, dict):
        audio_data = result.get("x")
        if audio_data is None:
            audio_data = result.get("audio")
        audio_sr = result.get("audio_sampling_rate", result.get("sample_rate", 48000))
    elif isinstance(result, tuple):
        audio_data = result[0]
        if len(result) > 1 and isinstance(result[1], int):
            audio_sr = result[1]
    else:
        audio_data = result

    if audio_data is None:
        return None, "❌ No audio data returned."

    if torch.is_tensor(audio_data):
        audio_np = audio_data.cpu().float().numpy()
    elif isinstance(audio_data, np.ndarray):
        audio_np = audio_data
    else:
        return None, f"❌ Unknown audio type: {type(audio_data)}"

    if audio_np.ndim == 3:
        audio_np = audio_np.squeeze(0)
    if audio_np.ndim == 2 and audio_np.shape[0] <= 2:
        audio_np = audio_np.T

    import soundfile as sf
    wav_path = tempfile.mktemp(suffix=f"_seed{seed}.wav")
    sf.write(wav_path, audio_np, int(audio_sr))

    mp3_path = wav_path.replace(".wav", ".mp3")
    try:
        subprocess.run([
            "ffmpeg", "-y", "-i", wav_path,
            "-codec:a", "libmp3lame", "-b:a", "192k", mp3_path
        ], check=True, capture_output=True)
        out_path = mp3_path if os.path.exists(mp3_path) and os.path.getsize(mp3_path) > 0 else wav_path
    except Exception:
        out_path = wav_path

    del audio_data, audio_np
    gc.collect()
    torch.cuda.empty_cache()

    return out_path, seed


# ==== MULTI-TRACK WRAPPER — uses gr.update() to only touch active tracks ====
@torch.inference_mode()
def generate_music(
    lyrics, caption, duration, seed,
    num_steps, guidance_scale, temperature,
    bpm, keyscale, timesignature, language,
    negative_prompt, audio_scale, shift, infer_method,
    audio_ref, audio_ref_timbre, audio_task, num_tracks,
    progress=gr.Progress(),
):
    try:
        n = max(1, min(4, int(num_tracks)))
        base_seed = int(seed) if seed is not None and int(seed) >= 0 else random.randint(0, 2**32 - 1)

        outputs = [None, None, None, None]
        statuses = []

        for i in range(n):
            track_seed = base_seed + i
            label = f"Track {i+1}/{n}"
            progress(i / n, desc=f"{label}: starting...")
            print(f"\n>>> {label} (seed={track_seed})")
            sys.stdout.flush()

            try:
                result_path, used_seed = _generate_single(
                    lyrics, caption, duration, track_seed,
                    num_steps, guidance_scale, temperature,
                    bpm, keyscale, timesignature, language,
                    negative_prompt, audio_scale, shift, infer_method,
                    audio_ref, audio_ref_timbre, audio_task,
                    progress_fn=progress,
                    progress_offset=i / n,
                    progress_scale=1.0 / n,
                    track_label=label,
                )
                if result_path is None:
                    statuses.append(f"❌ {label}: {used_seed}")
                else:
                    outputs[i] = result_path
                    statuses.append(f"✅ {label}: seed={used_seed}")
                    print(f"  ✅ {label} saved: {result_path}")
                    sys.stdout.flush()
            except Exception as e:
                traceback.print_exc()
                gc.collect()
                torch.cuda.empty_cache()
                statuses.append(f"❌ {label}: {str(e)}")

        progress(1.0, desc="Done!")
        status_text = "\n".join(statuses)

        # Use gr.update() — only set value for tracks we used, hide unused ones
        results = []
        for i in range(4):
            if i < n:
                results.append(gr.update(value=outputs[i], visible=True))
            else:
                results.append(gr.update(value=None, visible=False))
        results.append(status_text)
        return results

    except Exception as e:
        traceback.print_exc()
        gc.collect()
        torch.cuda.empty_cache()
        return [gr.update(value=None, visible=False) for _ in range(4)] + [f"❌ Error: {str(e)}"]


# ==== FASTAPI API SERVER (Ngrok 원격 접속용 — Gradio UI 대체) ====
import json as _json
import uuid as _uuid
import threading
import os
import sys
import gc
import traceback
import psutil
import torch
from typing import Optional
from fastapi import FastAPI, Form, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title="Daon Music Studio API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

_tasks = {}
_lock = threading.Lock()


@app.get("/health")
def api_health():
    try:
        vram = torch.cuda.mem_get_info()[0] / 1024**3
    except Exception:
        vram = 0.0
    try:
        ram_avail = psutil.virtual_memory().available / 1024**3
    except Exception:
        ram_avail = 0.0
    return {
        "status": "ok",
        "model": "다온 음악 생성 스튜디오",
        "vram_free_gb": round(vram, 2),
        "ram_free_gb": round(ram_avail, 2)
    }


def _run_task(task_id, params, audio_ref_path, audio_ref_timbre_path):
    with _lock:
        task = _tasks[task_id]
        task["status"] = "processing"
        
        n = max(1, min(4, int(params.get("num_tracks", 1))))
        base_seed = int(params["seed"])
        if base_seed < 0:
            import random
            base_seed = random.randint(0, 2**32 - 1)
            
        results = []
        statuses = []
        
        for i in range(n):
            track_seed = base_seed + i
            label = f"Track {i+1}/{n}"
            
            def prog(frac, desc=""):
                total_frac = (i + frac) / n
                task["progress"] = round(total_frac * 100, 1)
                if desc:
                    task["desc"] = desc
                else:
                    task["desc"] = f"{label}: Generating..."

            task["desc"] = f"{label}: Starting..."
            print(f"\n>>> FastAPI Task {task_id} | {label} (seed={track_seed})")
            sys.stdout.flush()
            
            try:
                result_path, used_seed = _generate_single(
                    lyrics=params["lyrics"],
                    caption=params["caption"],
                    duration=params["duration"],
                    seed=track_seed,
                    num_steps=params["num_steps"],
                    guidance_scale=params["guidance_scale"],
                    temperature=params["temperature"],
                    bpm=params["bpm"],
                    keyscale=params["keyscale"],
                    timesignature=params["timesignature"],
                    language=params["language"],
                    negative_prompt=params["negative_prompt"],
                    audio_scale=params["audio_scale"],
                    shift=params["shift"],
                    infer_method=params["infer_method"],
                    audio_ref=audio_ref_path,
                    audio_ref_timbre=audio_ref_timbre_path,
                    audio_task=params["audio_task"],
                    progress_fn=prog,
                    progress_offset=0.0,
                    progress_scale=1.0,
                    track_label=label,
                )
                if result_path:
                    results.append({
                        "track_num": i + 1,
                        "seed": used_seed,
                        "file_path": result_path,
                        "filename": os.path.basename(result_path)
                    })
                    statuses.append(f"✅ {label}: seed={used_seed}")
                    print(f"  ✅ {label} saved: {result_path}")
                else:
                    statuses.append(f"❌ {label}: Generation failed (returned None)")
            except Exception as e:
                traceback.print_exc()
                statuses.append(f"❌ {label}: {str(e)}")
                gc.collect()
                torch.cuda.empty_cache()
            sys.stdout.flush()
                
        # Clean up temporary uploaded files
        if audio_ref_path and os.path.exists(audio_ref_path):
            try:
                os.remove(audio_ref_path)
            except:
                pass
        if audio_ref_timbre_path and os.path.exists(audio_ref_timbre_path):
            try:
                os.remove(audio_ref_timbre_path)
            except:
                pass
                
        if len(results) > 0:
            task["status"] = "completed"
            task["results"] = results
            task["progress"] = 100.0
            task["desc"] = "\n".join(statuses)
            print(f"✅ Task {task_id} completed: {len(results)}/{n} tracks generated.")
        else:
            task["status"] = "error"
            task["error"] = "\n".join(statuses)
            task["desc"] = "Generation failed."
            print(f"❌ Task {task_id} failed.")
        sys.stdout.flush()


@app.post("/generate")
async def api_generate(
    lyrics: str = Form("[Instrumental]"),
    caption: str = Form("pop, upbeat, energetic"),
    duration: int = Form(60),
    seed: int = Form(-1),
    num_steps: int = Form(8),
    guidance_scale: float = Form(7.0),
    temperature: float = Form(1.0),
    bpm: str = Form(""),
    keyscale: str = Form(""),
    timesignature: str = Form(""),
    language: str = Form(""),
    negative_prompt: str = Form(""),
    audio_scale: float = Form(1.0),
    shift: float = Form(1.0),
    infer_method: str = Form("ode"),
    audio_task: str = Form("Text (Lyrics) ➔ Audio"),
    num_tracks: int = Form(1),
    audio_ref: Optional[UploadFile] = File(None),
    audio_ref_timbre: Optional[UploadFile] = File(None),
):
    task_id = str(_uuid.uuid4())
    _tasks[task_id] = {
        "status": "queued",
        "progress": 0.0,
        "desc": "대기 중...",
        "results": [],
        "error": None,
    }
    
    audio_ref_path = None
    if audio_ref and audio_ref.filename:
        import tempfile
        suffix = os.path.splitext(audio_ref.filename)[1] or ".wav"
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
            tmp.write(await audio_ref.read())
            audio_ref_path = tmp.name
            
    audio_ref_timbre_path = None
    if audio_ref_timbre and audio_ref_timbre.filename:
        import tempfile
        suffix = os.path.splitext(audio_ref_timbre.filename)[1] or ".wav"
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
            tmp.write(await audio_ref_timbre.read())
            audio_ref_timbre_path = tmp.name
            
    params = {
        "lyrics": lyrics,
        "caption": caption,
        "duration": duration,
        "seed": seed,
        "num_steps": num_steps,
        "guidance_scale": guidance_scale,
        "temperature": temperature,
        "bpm": bpm,
        "keyscale": keyscale,
        "timesignature": timesignature,
        "language": language,
        "negative_prompt": negative_prompt,
        "audio_scale": audio_scale,
        "shift": shift,
        "infer_method": infer_method,
        "audio_task": audio_task,
        "num_tracks": num_tracks,
    }
    
    threading.Thread(
        target=_run_task, 
        args=(task_id, params, audio_ref_path, audio_ref_timbre_path), 
        daemon=True
    ).start()
    
    return {"task_id": task_id, "status": "queued"}


@app.get("/status/{task_id}")
def api_status(task_id: str):
    task = _tasks.get(task_id)
    if not task:
        return JSONResponse({"error": "Task not found"}, status_code=404)
    r = {
        "task_id": task_id,
        "status": task["status"],
        "progress": task["progress"],
        "desc": task["desc"]
    }
    if task["status"] == "completed":
        r["results"] = [
            {"track_num": item["track_num"], "seed": item["seed"]}
            for item in task["results"]
        ]
    elif task["status"] == "error":
        r["error"] = task["error"]
    return r


@app.get("/result/{task_id}/{track_num}")
def api_result(task_id: str, track_num: int):
    task = _tasks.get(task_id)
    if not task:
        return JSONResponse({"error": "Task not found"}, status_code=404)
    if task["status"] != "completed":
        return JSONResponse({"error": "Not ready", "status": task["status"]}, status_code=400)
        
    track_result = None
    for r in task.get("results", []):
        if r["track_num"] == track_num:
            track_result = r
            break
            
    if not track_result or not os.path.exists(track_result["file_path"]):
        return JSONResponse({"error": f"Result file for track {track_num} not found"}, status_code=404)
        
    fn = track_result["filename"]
    mt = "audio/mpeg" if fn.endswith(".mp3") else "audio/wav"
    return FileResponse(track_result["file_path"], media_type=mt, filename=fn)


# ==== NGROK SETUP & LAUNCH ====
_ngrok_token = os.environ.get("NGROK_TOKEN", "")
if _ngrok_token:
    from pyngrok import ngrok
    ngrok.set_auth_token(_ngrok_token)
    for _tun in ngrok.get_tunnels():
        ngrok.disconnect(_tun.public_url)
    _pub = ngrok.connect(8000).public_url
    print(f"\n{'='*55}")
    print(f"\U0001f680 NGROK URL: {_pub}")
    print(f"{'='*55}")
else:
    print("\n⚠️ NGROK_TOKEN not set! Local only: http://localhost:8000")

print(f"\nAPI Endpoints:")
print(f"  GET  /health                  ➔ Server health check")
print(f"  POST /generate                ➔ Start music generation")
print(f"  GET  /status/{{task_id}}        ➔ Check progress")
print(f"  GET  /result/{{task_id}}/{{num}}  ➔ Download audio track")
print(f"\nStarting API server on port 8000...")
sys.stdout.flush()

uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


## 🚀 Cell 5 — Set Ngrok Token & Launch!


In [ ]:
# Cell 5: Set Ngrok Token & Launch!
import os

# ==========================================
# 🔥 여기에 본인의 Ngrok 토큰을 입력하세요!
# ==========================================
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"

if NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("❌ ERROR: Ngrok 토큰을 입력해주세요!")
else:
    os.environ["NGROK_TOKEN"] = NGROK_TOKEN
    !cd /kaggle/working && python -u run_acestep_xl.py 2>&1


---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---